In [ ]:
# =========================
# STAGE 5 — PORTFOLIO SYNTHESIS
# =========================

import pandas as pd
import numpy as np
import os
from scipy import stats

ARTIFACTS_PATH = r"C:\Users\Harsh Prakash\Desktop\Lending Club Credit Risk Project\artifacts"
PROJECT_PATH   = r"C:\Users\Harsh Prakash\Desktop\Lending Club Credit Risk Project"

# =========================
# LOAD STAGE 3 OUTPUTS
# =========================

pd_test    = pd.read_csv(os.path.join(ARTIFACTS_PATH, "pd_test.csv"))['pd']   # P(Bad)
score_test = pd.read_csv(os.path.join(ARTIFACTS_PATH, "score_test.csv"))['score']
y_test     = pd.read_csv(os.path.join(ARTIFACTS_PATH, "y_test.csv")).squeeze()
loan_amnt  = pd.read_csv(os.path.join(ARTIFACTS_PATH, "loan_amnt_test.csv")).squeeze()

# =========================
# 1. LGD (CORRECT — CONDITIONAL ON DEFAULT)
# =========================

cols_needed = ['total_pymnt', 'funded_amnt', 'loan_status']

df_full = pd.read_csv(
    os.path.join(PROJECT_PATH, "loan.csv"),
    usecols=cols_needed,
    low_memory=True
)

defaulted = df_full[df_full['loan_status'].isin([
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Charged Off'
])].copy()

defaulted = defaulted[defaulted['funded_amnt'] > 0]

defaulted['LGD'] = (
    1 - defaulted['total_pymnt'] / defaulted['funded_amnt']
).clip(0, 1)

mean_lgd = defaulted['LGD'].mean()

print(f"\nMEAN LGD (Conditional): {mean_lgd:.4f}")

# =========================
# 2. BUILD PORTFOLIO
# =========================

df = pd.DataFrame({
    'score': score_test,
    'pd': pd_test,
    'actual': y_test,
    'ead': loan_amnt
})

# Expected Loss
df['EL'] = df['pd'] * mean_lgd * df['ead']

# =========================
# 3. REVENUE & PROFIT (RISK-BASED PRICING)
# =========================

# Load interest rates
rate_df = pd.read_csv(
    os.path.join(PROJECT_PATH, "loan.csv"),
    usecols=['int_rate']
)

# Clean interest rate (e.g., "13.56%" → 0.1356)
rate_df['int_rate'] = (
    rate_df['int_rate']
    .astype(str)
    .str.replace('%', '')
    .astype(float) / 100
)

# 🔴 CRITICAL: Align with test set
# Assumes Stage 1 preserved row order
df['rate'] = rate_df.iloc[y_test.index].values

# Revenue = loan-level pricing
df['revenue'] = df['ead'] * df['rate']

# Profit
df['profit'] = df['revenue'] - df['EL']

# =========================
# 4. CUT-OFF OPTIMIZATION (PROFIT)
# =========================

cutoffs = np.arange(350, 601, 10)
results = []

for cutoff in cutoffs:
    approved = df[df['score'] >= cutoff]

    if len(approved) == 0:
        continue

    results.append({
        'cutoff': cutoff,
        'approval_rate': len(approved) / len(df),
        'avg_pd': approved['pd'].mean(),
        'total_EL': approved['EL'].sum(),
        'total_revenue': approved['revenue'].sum(),
        'total_profit': approved['profit'].sum(),
        'num_loans': len(approved)
    })

cutoff_df = pd.DataFrame(results)

print("\nCUTOFF STRATEGY:")
print(cutoff_df)

# Optimal cutoff = max profit
best = cutoff_df.sort_values('total_profit', ascending=False).iloc[0]

print("\nOPTIMAL CUT-OFF (PROFIT MAX):")
print(best)

# =========================
# 5. APPROVED PORTFOLIO
# =========================

chosen_cutoff = int(best['cutoff'])
approved = df[df['score'] >= chosen_cutoff]

print(f"\nAPPROVED @ SCORE >= {chosen_cutoff}")
print(f"Approval Rate: {len(approved)/len(df):.2%}")
print(f"Avg PD: {approved['pd'].mean():.4f}")
print(f"Total EL: {approved['EL'].sum():,.0f}")
print(f"Total Profit: {approved['profit'].sum():,.0f}")

# =========================
# 6. RISK SEGMENTATION
# =========================

df['risk_band'] = pd.qcut(df['score'], 5, labels=[
    'Very High Risk',
    'High Risk',
    'Medium Risk',
    'Low Risk',
    'Very Low Risk'
])

segment = df.groupby('risk_band').agg(
    count=('score','count'),
    avg_pd=('pd','mean'),
    total_el=('EL','sum'),
    total_profit=('profit','sum')
).reset_index()

print("\nRISK SEGMENTS:")
print(segment)

# =========================
# 7. IFRS 9 (12M vs LIFETIME ECL)
# =========================

def ifrs9_stage(pd):
    if pd < 0.05:
        return 1
    elif pd < 0.20:
        return 2
    else:
        return 3

df['stage'] = df['pd'].apply(ifrs9_stage)

LIFETIME_MULTIPLIER = 2.5  # documented assumption

df['ECL'] = np.where(
    df['stage'] == 1,
    df['pd'] * mean_lgd * df['ead'],
    df['pd'] * LIFETIME_MULTIPLIER * mean_lgd * df['ead']
)

stage_summary = df.groupby('stage').agg(
    count=('score','count'),
    total_ECL=('ECL','sum')
).reset_index()

print("\nIFRS 9 STAGING:")
print(stage_summary)

# =========================
# 8. BASEL IRB (UNEXPECTED LOSS)
# =========================

pd_vals = df['pd'].clip(1e-6, 1 - 1e-6)

R = 0.03 * (1 - np.exp(-35 * pd_vals)) / (1 - np.exp(-35)) + \
    0.16 * (1 - (1 - np.exp(-35 * pd_vals)) / (1 - np.exp(-35)))

K = mean_lgd * (
    stats.norm.cdf(
        stats.norm.ppf(pd_vals) / np.sqrt(1 - R) +
        np.sqrt(R / (1 - R)) * stats.norm.ppf(0.999)
    ) - pd_vals
)

df['RWA'] = K * df['ead'] * 12.5

print("\nBASEL RWA (IRB APPROX):")
print(f"Total RWA: {df['RWA'].sum():,.0f}")

# =========================
# 9. PORTFOLIO SUMMARY
# =========================

print("\nPORTFOLIO SUMMARY:")
print(f"Total Loans: {len(df)}")
print(f"Avg PD: {df['pd'].mean():.4f}")
print(f"Total EL: {df['EL'].sum():,.0f}")
print(f"Total Profit: {df['profit'].sum():,.0f}")
print(f"EL % of EAD: {(df['EL'].sum()/df['ead'].sum()*100):.2f}%")

# =========================
# FINAL
# =========================

print("\n✅ STAGE 5 FINAL — RISK + PROFIT + IFRS9 + BASEL COMPLETE")



MEAN LGD (Conditional): 0.4704

CUTOFF STRATEGY:
    cutoff  approval_rate    avg_pd      total_EL  total_revenue  \
0      350       1.000000  0.162516  2.520739e+08   4.166798e+08   
1      360       1.000000  0.162516  2.520739e+08   4.166798e+08   
2      370       0.999964  0.162502  2.520484e+08   4.166664e+08   
3      380       0.999350  0.162298  2.516178e+08   4.164290e+08   
4      390       0.995823  0.161277  2.488731e+08   4.147631e+08   
5      400       0.985366  0.158637  2.410431e+08   4.094853e+08   
6      410       0.965751  0.154350  2.275036e+08   3.993459e+08   
7      420       0.934740  0.148439  2.090249e+08   3.840624e+08   
8      430       0.882421  0.139803  1.818870e+08   3.586347e+08   
9      440       0.819726  0.130658  1.550149e+08   3.300094e+08   
10     450       0.738721  0.119975  1.264009e+08   2.957064e+08   
11     460       0.650771  0.109275  1.016573e+08   2.616933e+08   
12     470       0.551538  0.097872  7.769563e+07   2.233713e+08  

C:\Users\Harsh Prakash\AppData\Local\Temp\ipykernel_29144\1910113649.py:151: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  segment = df.groupby('risk_band').agg(


In [1]:
#save portfolio to csv
df.to_csv(os.path.join(ARTIFACTS_PATH, "final_portfolio.csv"), index=False)

NameError: name 'df' is not defined